# Real ML Analysis for MDPI Paper

**Purpose:** Train four ML classifiers on the real JIRA defect dataset, compute SHAP values for the best model, and produce real figures for the MDPI paper.

**How to use:**
1. Click `Runtime > Run all` from the menu (or run cells one by one)
2. When prompted, upload your CSV file
3. Wait ~10-15 minutes for completion
4. Three output files will be saved: `shap_beeswarm.png`, `confusion_matrix.png`, `results.txt`
5. Download those files at the end

**What it does:**
- Loads your CSV
- Engineers features (binarizes Severity into high-risk vs low-risk target)
- Trains Random Forest, Gradient Boosting, Logistic Regression, SVM
- Computes Precision, Recall, F1, AUC-ROC for each
- Runs SHAP analysis on the best model
- Saves beeswarm plot and confusion matrix as PNG

**Important:** The numbers this produces are REAL. They may differ from what was previously in the paper. We will update the paper to match these real results.

## Step 1: Install required libraries

Most are pre-installed in Colab; we just need SHAP and imbalanced-learn.

In [ ]:
!pip install -q shap imbalanced-learn
print('Libraries installed.')

## Step 2: Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                              confusion_matrix, classification_report, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
import shap

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Libraries imported.')

## Step 3: Upload your CSV file

When the file picker appears, select your JIRA export CSV.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
print(f'Uploaded: {csv_filename}')

## Step 4: Load the data and inspect it

Verify the data loaded correctly before training anything.

In [ ]:
# Try common encodings since JIRA exports vary
for encoding in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
    try:
        df = pd.read_csv(csv_filename, encoding=encoding)
        print(f'Loaded with encoding: {encoding}')
        break
    except (UnicodeDecodeError, UnicodeError):
        continue

print(f'\nTotal rows: {len(df):,}')
print(f'Total columns: {len(df.columns)}')
print(f'\nColumn names found:')
for c in df.columns:
    print(f'  - {c}')
print(f'\nFirst 3 rows preview:')
df.head(3)

## Step 5: Check the Severity distribution

This tells us how skewed the data is and confirms severity values.

In [ ]:
print('Severity value counts:')
print(df['Severity'].value_counts(dropna=False))
print(f'\nMissing Severity values: {df["Severity"].isna().sum()}')
print(f'\nPercentage distribution:')
print((df['Severity'].value_counts(normalize=True, dropna=False) * 100).round(2))

## Step 6: Filter to closed/resolved defects and engineer features

We only train on defects with complete lifecycle records (creation + resolution).

In [ ]:
# Keep only rows where we have a severity label and a resolution
df_clean = df.dropna(subset=['Severity']).copy()
print(f'Rows after dropping null severity: {len(df_clean):,}')

# Parse dates
df_clean['Created'] = pd.to_datetime(df_clean['Created'], errors='coerce')
df_clean['Resolved'] = pd.to_datetime(df_clean['Resolved'], errors='coerce')
df_clean['Updated'] = pd.to_datetime(df_clean['Updated'], errors='coerce')

# Drop rows without a Created date (data quality)
df_clean = df_clean.dropna(subset=['Created'])
print(f'Rows after dropping null Created: {len(df_clean):,}')

# --- TARGET LABEL ---
# Binarize: High-risk = Critical, High, or Severe; Low-risk = Medium, Normal, Low, Minor
# Blanks excluded (filtered above via dropna)
high_risk_severities = ['Critical', 'High', 'Severe']
# Normalize case to handle any typos like 'critical' vs 'Critical'
df_clean['Severity_normalized'] = df_clean['Severity'].astype(str).str.strip().str.title()
df_clean['high_risk'] = df_clean['Severity_normalized'].isin(high_risk_severities).astype(int)

print(f'\nTarget label distribution (high-risk = 1):')
print(df_clean['high_risk'].value_counts())
print(f'High-risk percentage: {df_clean["high_risk"].mean()*100:.2f}%')

## Step 7: Feature engineering

Build features only from columns that actually exist in your data.

In [ ]:
# --- Days to Resolution ---
df_clean['days_to_resolution'] = (df_clean['Resolved'] - df_clean['Created']).dt.total_seconds() / 86400
# Replace negative or null with median (data quality fallback)
median_dtr = df_clean['days_to_resolution'].median()
df_clean['days_to_resolution'] = df_clean['days_to_resolution'].fillna(median_dtr).clip(lower=0)

# --- Defect age (for unresolved or as fallback) ---
df_clean['defect_age_days'] = (df_clean['Updated'] - df_clean['Created']).dt.total_seconds() / 86400
df_clean['defect_age_days'] = df_clean['defect_age_days'].fillna(median_dtr).clip(lower=0)

# --- Temporal features from Created ---
df_clean['created_month'] = df_clean['Created'].dt.month
df_clean['created_dow'] = df_clean['Created'].dt.dayofweek
df_clean['created_quarter'] = df_clean['Created'].dt.quarter

# --- Re-open Count ---
df_clean['reopen_count'] = pd.to_numeric(df_clean['Re-open counter'], errors='coerce').fillna(0)

# --- Has linked issues (count of linked items) ---
df_clean['has_linked_issues'] = df_clean['Linked Issues'].notna().astype(int)
df_clean['linked_issues_count'] = df_clean['Linked Issues'].fillna('').astype(str).str.count(',') + df_clean['has_linked_issues']

# --- IT Impacted Apps Count ---
df_clean['impacted_apps_count'] = df_clean['IT Impacted Applications'].fillna('').astype(str).str.count(',') + df_clean['IT Impacted Applications'].notna().astype(int)

# --- Labels count (proxy for tag richness) ---
df_clean['labels_count'] = df_clean['Labels'].fillna('').astype(str).str.count(',') + df_clean['Labels'].notna().astype(int)

print('Numeric features engineered.')
print(df_clean[['days_to_resolution', 'defect_age_days', 'reopen_count', 
                 'linked_issues_count', 'impacted_apps_count', 'labels_count',
                 'created_month', 'created_dow', 'created_quarter']].describe().round(2))

In [ ]:
# --- Component-level aggregate features (TEAM-LEVEL, no individuals) ---
# Take primary component (first one if multiple)
df_clean['primary_component'] = df_clean['Component/s'].fillna('Unknown').astype(str).str.split(',').str[0].str.strip()

# Component historical defect rate (component-level aggregate)
component_counts = df_clean.groupby('primary_component').size()
component_high_risk = df_clean.groupby('primary_component')['high_risk'].sum()
component_high_risk_rate = (component_high_risk / component_counts).fillna(0)
df_clean['component_high_risk_rate'] = df_clean['primary_component'].map(component_high_risk_rate)
df_clean['component_total_defects'] = df_clean['primary_component'].map(component_counts)

# Reporter aggregate (number of prior reports by same reporter — proxy for reporter experience)
reporter_counts = df_clean.groupby('Reporter').size()
df_clean['reporter_history_count'] = df_clean['Reporter'].map(reporter_counts).fillna(1)

print('Component and reporter aggregates engineered.')
print(f'Unique components: {df_clean["primary_component"].nunique()}')
print(f'Unique reporters: {df_clean["Reporter"].nunique()}')

In [ ]:
# --- Categorical features (one-hot or label-encoded) ---

# Priority
df_clean['Priority'] = df_clean['Priority'].fillna('Unknown').astype(str)

# Issue Type
df_clean['Issue Type'] = df_clean['Issue Type'].fillna('Unknown').astype(str)

# Bug Category
df_clean['Bug Category'] = df_clean['Bug Category'].fillna('Unknown').astype(str)

# Defect Type
df_clean['Defect Type'] = df_clean['Defect Type'].fillna('Unknown').astype(str)

# Status (filter context)
df_clean['Status'] = df_clean['Status'].fillna('Unknown').astype(str)

# Component (top-15 only, rest as 'Other' to keep dimensionality manageable)
top_components = df_clean['primary_component'].value_counts().head(15).index
df_clean['component_grouped'] = df_clean['primary_component'].where(
    df_clean['primary_component'].isin(top_components), 'Other'
)

# One-hot encode the categoricals
# NOTE: Priority is intentionally EXCLUDED from features.
# Priority is highly correlated with Severity (the prediction target) — including it
# would cause label leakage and inflate apparent performance. Reviewers in this subfield
# would flag this immediately. Removing Priority forces the model to find genuine
# predictive signal from features that are not proxies for the answer.
categorical_features = ['Issue Type', 'Bug Category', 'Defect Type', 'component_grouped']
df_encoded = pd.get_dummies(df_clean, columns=categorical_features, drop_first=True)

print(f'After one-hot encoding: {df_encoded.shape[1]} columns total')

## Step 8: Prepare feature matrix X and target y

In [ ]:
# Select final feature set
numeric_features = [
    'days_to_resolution', 'defect_age_days', 'reopen_count',
    'linked_issues_count', 'has_linked_issues', 'impacted_apps_count', 'labels_count',
    'created_month', 'created_dow', 'created_quarter',
    'component_high_risk_rate', 'component_total_defects', 'reporter_history_count'
]

# Get all one-hot columns from the categoricals
categorical_cols = [c for c in df_encoded.columns if any(c.startswith(prefix + '_') for prefix in categorical_features)]

feature_columns = numeric_features + categorical_cols
# Filter out any features that aren't actually in df_encoded (safety)
feature_columns = [c for c in feature_columns if c in df_encoded.columns]

# DEFENSIVE SAFEGUARD: explicitly remove any Priority and Severity columns
# to prevent label leakage even if encoding accidentally includes them
leakage_columns = [c for c in feature_columns if c.startswith('Priority') or c.startswith('Severity') or c.startswith('Bug Severity')]
if leakage_columns:
    print(f'WARNING: Removing leakage columns: {leakage_columns}')
    feature_columns = [c for c in feature_columns if c not in leakage_columns]

X = df_encoded[feature_columns].copy()
y = df_encoded['high_risk'].copy()

# Convert bool columns to int
for col in X.columns:
    if X[col].dtype == bool:
        X[col] = X[col].astype(int)

# Fill any remaining NaNs
X = X.fillna(0)

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Class balance: {y.mean()*100:.2f}% high-risk')
print(f'\nFinal feature list ({len(feature_columns)} features):')
for f in feature_columns:
    print(f'  - {f}')

## Step 9: Train/test split (80/20, stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Training set: {X_train.shape[0]:,} rows ({y_train.mean()*100:.2f}% high-risk)')
print(f'Test set:     {X_test.shape[0]:,} rows ({y_test.mean()*100:.2f}% high-risk)')

## Step 10: Apply SMOTE to training set only

This balances the classes without leaking synthetic data into the test set.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
print(f'Training set after SMOTE: {X_train_balanced.shape[0]:,} rows ({y_train_balanced.mean()*100:.2f}% high-risk)')

## Step 11: Train four models with simplified grid search

Full grid search would take 30+ minutes. This uses a focused, reduced grid that produces realistic results in a few minutes.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)  # 3-fold for speed

results = {}

# --- Random Forest ---
print('Training Random Forest...')
rf_params = {'n_estimators': [200], 'max_depth': [10, 20], 'min_samples_split': [5]}
rf = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'),
                  rf_params, cv=cv, scoring='f1', n_jobs=-1)
rf.fit(X_train_balanced, y_train_balanced)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
results['Random Forest'] = {
    'model': rf.best_estimator_,
    'y_pred': y_pred_rf, 'y_prob': y_prob_rf,
    'precision': precision_score(y_test, y_pred_rf),
    'recall': recall_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf),
    'auc': roc_auc_score(y_test, y_prob_rf)
}
print(f'  Best params: {rf.best_params_}')

# --- Gradient Boosting ---
print('Training Gradient Boosting...')
gb_params = {'n_estimators': [200], 'max_depth': [3, 5], 'learning_rate': [0.1]}
gb = GridSearchCV(GradientBoostingClassifier(random_state=RANDOM_STATE),
                  gb_params, cv=cv, scoring='f1', n_jobs=-1)
gb.fit(X_train_balanced, y_train_balanced)
y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]
results['Gradient Boosting'] = {
    'model': gb.best_estimator_,
    'y_pred': y_pred_gb, 'y_prob': y_prob_gb,
    'precision': precision_score(y_test, y_pred_gb),
    'recall': recall_score(y_test, y_pred_gb),
    'f1': f1_score(y_test, y_pred_gb),
    'auc': roc_auc_score(y_test, y_prob_gb)
}
print(f'  Best params: {gb.best_params_}')

# --- Logistic Regression ---
print('Training Logistic Regression...')
lr_params = {'C': [0.1, 1.0, 10.0], 'penalty': ['l2']}
lr = GridSearchCV(LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced'),
                  lr_params, cv=cv, scoring='f1', n_jobs=-1)
lr.fit(X_train_scaled, y_train_balanced)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
results['Logistic Regression'] = {
    'model': lr.best_estimator_,
    'y_pred': y_pred_lr, 'y_prob': y_prob_lr,
    'precision': precision_score(y_test, y_pred_lr),
    'recall': recall_score(y_test, y_pred_lr),
    'f1': f1_score(y_test, y_pred_lr),
    'auc': roc_auc_score(y_test, y_prob_lr)
}
print(f'  Best params: {lr.best_params_}')

# --- SVM ---
print('Training SVM (this is the slowest)...')
svm_params = {'C': [1.0], 'kernel': ['rbf'], 'gamma': ['scale']}
svm = GridSearchCV(SVC(random_state=RANDOM_STATE, probability=True, class_weight='balanced'),
                   svm_params, cv=cv, scoring='f1', n_jobs=-1)
svm.fit(X_train_scaled, y_train_balanced)
y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]
results['SVM'] = {
    'model': svm.best_estimator_,
    'y_pred': y_pred_svm, 'y_prob': y_prob_svm,
    'precision': precision_score(y_test, y_pred_svm),
    'recall': recall_score(y_test, y_pred_svm),
    'f1': f1_score(y_test, y_pred_svm),
    'auc': roc_auc_score(y_test, y_prob_svm)
}
print(f'  Best params: {svm.best_params_}')

print('\nAll models trained.')

## Step 12: Print and save the real Table 3 numbers

These are the numbers that go into the paper's Table 3 — whatever they are.

In [ ]:
results_df = pd.DataFrame({
    name: {'Precision': r['precision'], 'Recall': r['recall'], 'F1': r['f1'], 'AUC-ROC': r['auc']}
    for name, r in results.items()
}).T.round(3)

print('='*70)
print('TABLE 3 — Model Performance Comparison (REAL NUMBERS)')
print('='*70)
print(results_df.to_string())
print('='*70)

# Save to text file
with open('results.txt', 'w') as f:
    f.write('TABLE 3 - Model Performance Comparison\n')
    f.write('='*70 + '\n')
    f.write(results_df.to_string() + '\n')
    f.write('\n')
    f.write(f'Dataset size: {len(df_clean):,} rows\n')
    f.write(f'High-risk class: {y.mean()*100:.2f}%\n')
    f.write(f'Features used: {len(feature_columns)}\n')

# Best model
best_model_name = results_df['F1'].idxmax()
print(f'\nBest model by F1: {best_model_name} (F1={results_df.loc[best_model_name, "F1"]:.3f})')

## Step 13: Confusion Matrix figure for best model

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
best_pred = results[best_model_name]['y_pred']
cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low-risk', 'High-risk'])
disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

# Append to results.txt
with open('results.txt', 'a') as f:
    f.write(f'\nConfusion Matrix ({best_model_name}):\n')
    f.write(f'                  Predicted Low  Predicted High\n')
    f.write(f'Actual Low-risk:  {cm[0,0]:>13}  {cm[0,1]:>14}\n')
    f.write(f'Actual High-risk: {cm[1,0]:>13}  {cm[1,1]:>14}\n')
    f.write(f'\nClassification Report:\n')
    f.write(classification_report(y_test, best_pred, target_names=['Low-risk', 'High-risk']))

## Step 14: SHAP analysis on the best model

This is the main figure for the paper. SHAP values are computed on a test sample.

In [ ]:
best_model = results[best_model_name]['model']

# Sample test set for SHAP (full set can be slow for tree explainer on large data)
shap_sample_size = min(500, len(X_test))
X_shap = X_test.sample(n=shap_sample_size, random_state=RANDOM_STATE)

print(f'Computing SHAP values on {shap_sample_size} test samples...')

if best_model_name in ['Random Forest', 'Gradient Boosting']:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_shap)
    # For binary classification, shap_values may be a list of two arrays
    if isinstance(shap_values, list):
        shap_values_for_plot = shap_values[1]  # Class 1 (high-risk)
    elif len(shap_values.shape) == 3:
        shap_values_for_plot = shap_values[:, :, 1]
    else:
        shap_values_for_plot = shap_values
else:
    # For non-tree models, KernelExplainer (slower)
    X_shap_scaled = scaler.transform(X_shap)
    background = shap.sample(scaler.transform(X_train_balanced), 100, random_state=RANDOM_STATE)
    explainer = shap.KernelExplainer(best_model.predict_proba, background)
    shap_values = explainer.shap_values(X_shap_scaled, nsamples=100)
    shap_values_for_plot = shap_values[1] if isinstance(shap_values, list) else shap_values

print('SHAP values computed.')

## Step 15: Generate the SHAP beeswarm plot (Figure 1)

This is THE figure for the paper.

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values_for_plot, X_shap, max_display=15, show=False)
plt.title(f'SHAP Feature Importance — {best_model_name}', fontsize=12, pad=15)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: shap_beeswarm.png')

# Top features by mean absolute SHAP
mean_abs_shap = np.abs(shap_values_for_plot).mean(axis=0)
feature_importance_df = pd.DataFrame({
    'Feature': X_shap.columns,
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=False).head(15)

print('\nTop 15 features by SHAP importance:')
print(feature_importance_df.to_string(index=False))

# Save
with open('results.txt', 'a') as f:
    f.write('\n\nTop 15 features by SHAP importance:\n')
    f.write(feature_importance_df.to_string(index=False) + '\n')

## Step 16: Download all outputs

Three files: SHAP plot, confusion matrix, and the text file with all the numbers.

In [ ]:
from google.colab import files
files.download('shap_beeswarm.png')
files.download('confusion_matrix.png')
files.download('results.txt')
print('\nAll files downloaded. Share results.txt back to update the paper.')